# Bài học 18: Bảo mật Đại lý AI với Biên lai Mã hóa

## Sổ tay Thực hành

Sổ tay này hướng dẫn bốn bài tập:

1. **Ký biên lai đầu tiên của bạn** cho một cuộc gọi công cụ đại lý và xác thực nó.
2. **Thao túng biên lai** và xem quá trình xác thực thất bại.
3. **Xây dựng chuỗi ba biên lai** và xác nhận tính toàn vẹn của chuỗi.
4. **Đóng gói cuộc gọi công cụ Microsoft Agent Framework** để mỗi hành động phát ra một biên lai.

Tất cả các nguyên lý mật mã đều được nhập khẩu từ các thư viện được duy trì tốt (`pynacl` cho Ed25519, `jcs` cho JSON theo chuẩn RFC 8785, `hashlib` từ thư viện tiêu chuẩn Python cho SHA-256). Logic biên lai tự nó là Python giản đơn mà bạn có thể đọc và chỉnh sửa.

Chạy các ô theo thứ tự. Mỗi phần đều ngắn gọn và độc lập.


## Thiết lập

Cài đặt hai phụ thuộc. Cả hai đều có giấy phép mở (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Tiện ích Hỗ trợ

Hai tiện ích này xử lý mã hóa base64url (không có đệm) và băm SHA-256 của các đối tượng tùy ý. Chúng giữ cho phần còn lại của sổ tay tập trung vào logic hóa đơn chính.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Ký biên lai đầu tiên của bạn

Hãy tưởng tượng đại lý của chúng ta cho **Contoso Travel** vừa tra cứu các chuyến bay từ Sydney đến Los Angeles cho một khách hàng. Chúng ta muốn lưu lại cuộc gọi công cụ này dưới dạng một biên lai đã ký để một kiểm toán viên tương lai có thể xác minh mà không cần tin tưởng chúng ta.

### Bước 1.1: Tạo khóa ký

Trong môi trường sản xuất, khóa ký của đại lý sẽ được lưu trong một mô-đun bảo mật phần cứng (HSM), Azure Key Vault hoặc một kho lưu trữ được bảo vệ tương tự. Với bài học này, chúng ta tạo một khóa mới trong bộ nhớ.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Bước 1.2: Xây dựng phần payload của biên nhận

Payload chứa tất cả những gì chúng ta muốn biên nhận xác nhận: ai đã thực hiện, công cụ nào, với những đối số gì, kết quả ra sao, theo chính sách nào, và khi nào. Chúng tôi băm các đối số và kết quả thay vì đưa chúng trực tiếp vào để biên nhận không làm rò rỉ nội dung nhạy cảm.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Bước 1.3: Ký và tập hợp biên lai

Ba bước:

1. Chuẩn hóa dữ liệu bằng JCS để hai triển khai tạo ra cùng một biên lai logic thì tạo ra các byte giống hệt nhau.
2. Băm các byte chuẩn hóa bằng SHA-256.
3. Ký băm bằng khóa riêng Ed25519.

Chữ ký sau đó được đính kèm vào dữ liệu gốc để tạo ra biên lai cuối cùng.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Bước 1.4: Xác minh biên nhận

Quá trình xác minh đảo ngược quy trình. Chúng tôi loại bỏ chữ ký, tính lại hàm băm chuẩn, và kiểm tra chữ ký với khóa công khai trong biên nhận.

Một kiểm toán viên thực hiện xác minh này không cần gì từ chúng tôi ngoài chính biên nhận. Không cần gọi dịch vụ, không cần tra cứu thư mục khóa, không cần tin cậy.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Bạn sẽ thấy `Receipt is valid: True`. Đại lý đã tạo ra bản ghi kiểm tra được ký mã hóa đầu tiên của nó.


## Phần 2: Can thiệp vào biên nhận

Mục đích của biên nhận là để ngăn chặn việc chỉnh sửa không được phát hiện. Hãy chứng minh điều đó.

Chúng ta sẽ sửa đổi đúng một ký tự của biên nhận và quan sát việc xác thực thất bại.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Điều gì vừa xảy ra?

Khi chúng ta thay đổi `policy_id`, các byte chuẩn đã thay đổi. Băm SHA-256 của những byte đó cũng thay đổi. Chữ ký (được tạo dựa trên băm ban đầu) không còn khớp với băm mới. Việc xác thực trả về chính xác là `False`.

Không có cách nào để chỉnh sửa bất kỳ trường nào trong phiếu biên nhận mà vẫn xác thực được, trừ khi kẻ tấn công có chìa khóa riêng. Miễn là chìa khóa riêng được lưu trong kho khóa và chìa khóa công khai được công bố, việc giả mạo là không thể giấu được.

Hãy thử tự mình: thay đổi `tool_name` hoặc `agent_id` hoặc `timestamp` trong ô trên và chạy lại. Mỗi thay đổi sẽ tạo ra một phiếu biên nhận không hợp lệ.


## Phần 3: Nối các biên nhận với nhau

Một biên nhận đơn lẻ bảo vệ một hành động. Hầu hết các tác nhân thực hiện nhiều hành động. Để làm cho toàn bộ chuỗi không thể giả mạo, chúng ta liên kết mỗi biên nhận với biên nhận trước đó bằng cách bao gồm băm của biên nhận trước trong phần payload của biên nhận mới.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Nếu ai đó loại bỏ hoặc thay đổi thứ tự một biên nhận, chuỗi sẽ bị đứt ngay tại điểm đó. Việc xác minh bất kỳ biên nhận nào sau đó sẽ thất bại vì `previous_receipt_hash` không còn khớp với băm thực tế của biên nhận trước đó nữa.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Bây giờ phá chuỗi bằng cách làm giả biên nhận ở giữa và kiểm tra lại. Biên nhận bị làm giả sẽ không qua được kiểm tra chữ ký, VÀ biên nhận tiếp theo không qua được kiểm tra liên kết chuỗi (bởi vì `previous_receipt_hash` của nó không còn khớp với hàm băm của biên nhận giữa đã bị sửa đổi).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Biên lai 0 vẫn xác minh được (nó không bị thay đổi và không có biên lai trước để phụ thuộc). Biên lai 1 không qua kiểm tra chữ ký vì chúng ta đã thay đổi `tool_args_hash`. Biên lai 2 không qua kiểm tra liên kết chuỗi vì `previous_receipt_hash` của nó được tính dựa trên biên lai 1 gốc (bây giờ đã bị chỉnh sửa).

Ngay cả khi kẻ tấn công ký lại biên lai 1 đã chỉnh sửa (điều mà họ không thể làm được nếu không có khóa riêng), sự không khớp liên kết chuỗi trong biên lai 2 vẫn sẽ phơi bày sự giả mạo. Để che giấu sự thay đổi, kẻ tấn công sẽ phải ký lại tất cả các biên lai kể từ điểm chỉnh sửa trở đi, điều này đòi hỏi phải sở hữu khóa riêng.


## Phần 4: Bao bọc cuộc gọi công cụ đại lý với việc ký biên nhận

Trong một triển khai thực tế, bạn không muốn mỗi tác giả đại lý phải nhớ gọi `make_receipt`. Bạn muốn việc ký biên nhận được thực hiện tự động cho mỗi lần gọi công cụ.

Dưới đây là mẫu đơn giản nhất: một lớp bao bọc nhận bất kỳ hàm công cụ có thể gọi được nào và trả về phiên bản phát sinh biên nhận của nó. Điều này thích ứng với bất kỳ khung đại lý nào, bao gồm cả Microsoft Agent Framework (`agent_framework.azure`).

Nếu bạn chưa thiết lập dự án Azure AI Foundry, mô phỏng cục bộ bên dưới vẫn minh họa mẫu này.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Tích hợp với Microsoft Agent Framework

Bộ bao `ReceiptedTool` ở trên không phụ thuộc vào framework nào. Để sử dụng nó bên trong một agent được xây dựng với Microsoft Agent Framework, đăng ký hàm được bao quanh này như một công cụ. Một phác thảo (bạn sẽ thay thế bản mô phỏng bằng một đăng ký công cụ Azure AI Foundry thực tế):

```python
# Pseudocode hiển thị hình dạng tích hợp.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Bạn là một đại lý Du lịch Contoso ...",
#     tools=[receipted_lookup],   # công cụ đã được bọc, không phải hàm thô
# )
# response = agent.run("Tìm chuyến bay từ Sydney đến Los Angeles vào tháng Sáu.")
#
# # Sau khi chạy, mỗi lần gọi công cụ mà đại lý thực hiện đều có biên nhận đã ký:
# audit_chain = receipted_lookup.receipts
```

Agent framework không cần biết gì về biên nhận. Việc ký biên nhận được bao quanh quanh công cụ, không được gắn chặt vào framework. Đây là cách bạn thêm nguồn gốc vào mã agent hiện có mà không phải viết lại agent.


## Ôn tập và thử thách nâng cao

Bạn đã:

- Tạo cặp khóa Ed25519.
- Xây dựng và ký một biên nhận cho cuộc gọi công cụ đại lý.
- Xác minh biên nhận ngoại tuyến chỉ bằng khóa công khai.
- Thao túng một biên nhận và quan sát việc xác minh thất bại.
- Xây dựng một chuỗi băm liên kết gồm ba biên nhận.
- Thao túng phần giữa của chuỗi và quan sát cả lỗi chữ ký và lỗi liên kết chuỗi.
- Bao bọc một chức năng công cụ với việc ký biên nhận tự động.

**Thử thách nâng cao.** Mở rộng schema của biên nhận với trường `request_id` (một UUID dùng cho truy vết phân tán). Cập nhật `make_receipt` để bao gồm trường này, và xác nhận biên nhận vẫn xác minh được từ đầu đến cuối. Sau đó sửa đổi trường này sau khi ký và xác nhận việc xác minh thất bại. Điều này buộc bạn phải hiểu sâu cách mỗi byte trong mã hóa chuẩn đóng góp vào chữ ký.

**Ranh giới quan trọng.** Biên nhận chứng minh ba điều và chỉ ba điều: nguồn gốc (khóa này đã ký nội dung này), tính toàn vẹn (nội dung không thay đổi kể từ khi ký), và thứ tự (biên nhận này được tạo sau biên nhận kia). Chúng không chứng minh rằng hành động của đại lý là đúng, rằng chính sách được nêu trong `policy_id` thực sự đã được đánh giá, hoặc rằng đại lý đã tuân thủ mọi quy tắc. Biên nhận chỉ là nền tảng. Quản trị là hệ thống bạn xây dựng trên đó.

Đọc lại README bài học với ranh giới đó trong đầu. Sai lầm phổ biến nhất các nhóm mắc phải với biên nhận là nghĩ "chúng ta có biên nhận" tức là "chúng ta được quản trị." Không phải vậy. Biên nhận làm cho hành vi đại lý có thể kiểm toán được. Nó không làm cho hành vi đó trở nên đúng đắn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
